# 🩺 VERA Phase 2 - Training on Google Colab

This notebook trains the VERA model on Google Colab's free GPU and saves the trained weights for local use.

**Runtime:** ~4-6 hours on Colab GPU (T4)

## Setup Instructions
1. Runtime → Change runtime type → GPU (T4)
2. Run all cells
3. Download trained models at the end
4. Copy models to your local `models/checkpoints/` folder

## Step 1: Clone Repository & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone your repository (replace with your actual repo URL)
!git clone https://github.com/YOUR_USERNAME/VERA.git
%cd VERA/phase2_production

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q segmentation-models-pytorch
!pip install -q albumentations
!pip install -q timm
!pip install -q opencv-python-headless
!pip install -q scikit-learn scikit-image
!pip install -q pandas numpy matplotlib seaborn
!pip install -q PyYAML tqdm tensorboard

## Step 2: Download Datasets

In [ ]:
# Download APTOS 2019 dataset from Kaggle
# You need to upload your kaggle.json first

from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download APTOS dataset
!kaggle competitions download -c aptos2019-blindness-detection
!unzip -q aptos2019-blindness-detection.zip -d data/raw/aptos_2019/

# Organize files
!mkdir -p data/raw/aptos_2019/images
!mv data/raw/aptos_2019/train_images/*.png data/raw/aptos_2019/images/
!mv data/raw/aptos_2019/train.csv data/raw/aptos_2019/

In [ ]:
# Download vessel segmentation datasets (DRIVE)
!wget -q https://www.isi.uu.nl/Research/Databases/DRIVE/download.php -O drive.zip
!unzip -q drive.zip -d data/raw/DRIVE/

## Step 3: Train Vessel Segmentation Model

In [ ]:
# Train vessel segmenter (lightweight, ~30 minutes)
!python scripts/train_vessel_segmenter.py \
    --config configs/config.yaml \
    --epochs 30 \
    --batch-size 8 \
    --gpu 0

## Step 4: Cache Vessel Maps

In [ ]:
# Pre-compute vessel maps for faster training
!python scripts/cache_vessel_maps.py \
    --input data/raw/aptos_2019/images \
    --output data/vessel_cache \
    --checkpoint models/checkpoints/vessel_segmenter.pth

## Step 5: Train DR Classifier (Main Model)

In [ ]:
# Train DR classifier with attention-gated fusion
# This takes ~3-4 hours on Colab GPU
!python scripts/train.py \
    --config configs/config.yaml \
    --epochs 50 \
    --batch-size 16 \
    --gpu 0

## Step 6: Evaluate Model

In [ ]:
# Evaluate the trained model
!python scripts/evaluate.py \
    --checkpoint models/checkpoints/best_model.pth \
    --output results/

In [ ]:
# Display results
import json
from IPython.display import Image, display

# Show metrics
with open('results/metrics.json', 'r') as f:
    metrics = json.load(f)

print("="*50)
print("FINAL MODEL PERFORMANCE")
print("="*50)
print(f"Quadratic Weighted Kappa: {metrics['qwk']:.4f}")
print(f"Overall Accuracy: {metrics['accuracy']:.4f}")
print(f"AUC (OVR): {metrics['auc_ovr']:.4f}")
print(f"MAE: {metrics['mae']:.4f}")
print("="*50)

# Show confusion matrix
display(Image('results/confusion_matrix.png'))

## Step 7: Download Trained Models

In [ ]:
# Package models for download
!mkdir -p trained_models
!cp models/checkpoints/best_model.pth trained_models/
!cp models/checkpoints/vessel_segmenter.pth trained_models/
!cp results/metrics.json trained_models/

# Create zip file
!zip -r trained_models.zip trained_models/

print("✅ Models packaged successfully!")
print("📦 Download 'trained_models.zip' and extract to your local machine")

In [ ]:
# Download to your computer
from google.colab import files
files.download('trained_models.zip')

## Step 8: Test Single Image (Optional)

In [ ]:
# Quick test with a single image
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '.')

from src.models.classification import DRClassifier
from src.models.vessel_segmentation import UNetVesselSegmenter
from src.data.preprocessing import FundusPreprocessor
from src.utils.config import load_config

# Load models
config = load_config('configs/config.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load vessel segmenter
vessel_segmenter = UNetVesselSegmenter(
    encoder_name='resnet34',
    pretrained=False
).to(device)

vessel_checkpoint = torch.load('trained_models/vessel_segmenter.pth')
vessel_segmenter.load_state_dict(vessel_checkpoint['model_state_dict'])
vessel_segmenter.eval()

# Load DR classifier
dr_model = DRClassifier(
    backbone_name='resnet50',
    fusion_strategy='attention_gated',
    num_classes=5,
    pretrained=False
).to(device)

dr_checkpoint = torch.load('trained_models/best_model.pth')
dr_model.load_state_dict(dr_checkpoint['model_state_dict'])
dr_model.eval()

print("✅ Models loaded successfully!")

In [ ]:
# Test with an image
image_path = 'data/raw/aptos_2019/images/0005cfc8afb68.png'  # Change to your image
image = cv2.imread(image_path)

# Preprocess
preprocessor = FundusPreprocessor(
    target_size=(512, 512),
    ben_graham_enabled=True,
    clahe_enabled=True
)
processed = preprocessor.preprocess(image)

# Convert to tensor
image_tensor = torch.from_numpy(processed).permute(2, 0, 1).float().unsqueeze(0).to(device)

# Segment vessels
with torch.no_grad():
    vessel_output = vessel_segmenter(image_tensor)
    vessel_map = (torch.sigmoid(vessel_output) > 0.5).float()

# Predict DR grade
with torch.no_grad():
    logits = dr_model(image_tensor, vessel_map)
    probs = torch.softmax(logits, dim=1)
    prediction = logits.argmax(dim=1).item()
    confidence = probs[0, prediction].item()

# Display results
grade_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Image', fontsize=14)
axes[0].axis('off')

axes[1].imshow(vessel_map.squeeze().cpu().numpy(), cmap='gray')
axes[1].set_title('Vessel Segmentation', fontsize=14)
axes[1].axis('off')

axes[2].bar(grade_names, probs.squeeze().cpu().numpy())
axes[2].set_ylabel('Probability')
axes[2].set_title(f'Prediction: Grade {prediction} ({grade_names[prediction]}) - {confidence:.2%}', fontsize=14, fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🎯 Predicted: Grade {prediction} - {grade_names[prediction]}")
print(f"📊 Confidence: {confidence:.2%}")

## 📝 Instructions for Local Use

After downloading `trained_models.zip`:

1. Extract the zip file
2. Copy files to your local project:
   ```bash
   cp trained_models/best_model.pth VERA/phase2_production/models/checkpoints/
   cp trained_models/vessel_segmenter.pth VERA/phase2_production/models/checkpoints/
   ```
3. Run the web app:
   ```bash
   cd VERA/phase2_production
   streamlit run web_app/app.py
   ```
4. Upload retinopathy images and get predictions!

**No training needed on your local machine!** ✨